# 🌍 Macro & Market Dashboard

Economic indicators, market overview, and sector analysis via OpenBB.

**Data sources:** FRED (if API key), yfinance (index/world data), OpenBB economy modules.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timedelta

from openbb import obb
from src.data_engine import get_price_history, get_quote

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print('✅ Ready.')
print(f'Date: {datetime.now().strftime("%Y-%m-%d")}')

## 1. Major Indices Overview

In [ ]:
INDICES = {
    'S&P 500': '^GSPC',
    'NASDAQ': '^IXIC',
    'Dow Jones': '^DJI',
    'Russell 2000': '^RUT',
    'VIX': '^VIX',
}

print(f"{'Index':<20} {'Price':>10} {'Change':>10} {'% Change':>10}")
print('-' * 52)

for name, symbol in INDICES.items():
    try:
        quote = get_quote(symbol)
        if not quote.empty and 'close' in quote.columns:
            price = quote.iloc[0]['close']
            prev = quote.iloc[0].get('prev_close', price)
            change = price - prev if prev else 0
            pct = (change / prev * 100) if prev and prev != 0 else 0
            print(f"{name:<20} ${price:>9,.2f} {change:>+9,.2f} {pct:>+9.2f}%")
        else:
            print(f"{name:<20} {'—':>10}")
    except Exception as e:
        print(f"{name:<20} {'Error':>10} — {e}")

## 2. S&P 500 Performance (1 Year)

In [ ]:
spx = get_price_history('^GSPC', start=(datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d'))

if not spx.empty and 'close' in spx.columns:
    fig, ax = plt.subplots(figsize=(14, 6))
    if 'date' in spx.columns:
        spx['date'] = pd.to_datetime(spx['date'])
        spx = spx.set_index('date')
    
    ax.plot(spx.index, spx['close'], linewidth=1.5, color='navy')
    ax.fill_between(spx.index, spx['close'].min(), spx['close'], alpha=0.3, color='navy')
    
    # Annotate current level
    current = spx['close'].iloc[-1]
    ytd_start = spx.loc[spx.index >= f'{datetime.now().year}-01-01', 'close']
    if len(ytd_start) > 0:
        ytd_return = (current / ytd_start.iloc[0] - 1) * 100
    else:
        ytd_return = 0
    
    ax.set_title(f'S&P 500 — ${current:,.0f} (YTD: {ytd_return:+.1f}%)')
    ax.set_ylabel('Price')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No S&P 500 data.')

## 3. Tracked Portfolio Performance

In [ ]:
PORTFOLIO = ['NVDA', 'AVGO', 'ORCL']

fig, ax = plt.subplots(figsize=(14, 7))

start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

for ticker in PORTFOLIO:
    try:
        hist = get_price_history(ticker, start=start_date)
        if not hist.empty and 'close' in hist.columns:
            if 'date' in hist.columns:
                hist['date'] = pd.to_datetime(hist['date'])
                hist = hist.set_index('date')
            # Normalize to 100
            normalized = hist['close'] / hist['close'].iloc[0] * 100
            ax.plot(hist.index, normalized, linewidth=2, label=ticker)
    except Exception as e:
        print(f"  {ticker}: {e}")

# Compare to S&P 500
try:
    spx = get_price_history('^GSPC', start=start_date)
    if not spx.empty:
        if 'date' in spx.columns:
            spx['date'] = pd.to_datetime(spx['date'])
            spx = spx.set_index('date')
        normalized = spx['close'] / spx['close'].iloc[0] * 100
        ax.plot(spx.index, normalized, linewidth=1.5, color='gray', linestyle='--', alpha=0.7, label='S&P 500')
except Exception:
    pass

ax.set_title('Portfolio vs S&P 500 — 1 Year (Normalized to 100)')
ax.set_ylabel('Return (Base = 100)')
ax.axhline(y=100, color='black', linestyle='-', alpha=0.3)
ax.legend(loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Market Breadth (Gainers & Losers)

In [ ]:
print('### Top Gainers (Today)')
try:
    gainers = obb.equity.discovery.gainers(provider='yfinance')
    g_df = gainers.to_df() if gainers else pd.DataFrame()
    if not g_df.empty:
        display(g_df[['symbol', 'name', 'price', 'change_percent']].head(10))
    else:
        print('  No data returned.')
except Exception as e:
    print(f'  Not available via yfinance: {e}')
    print('  (Try FMP provider with API key)')

print()
print('### Top Losers (Today)')
try:
    losers = obb.equity.discovery.losers(provider='yfinance')
    l_df = losers.to_df() if losers else pd.DataFrame()
    if not l_df.empty:
        display(l_df[['symbol', 'name', 'price', 'change_percent']].head(10))
    else:
        print('  No data returned.')
except Exception as e:
    print(f'  Not available via yfinance: {e}')

## 5. Treasury Yields (FRED via OpenBB)

In [ ]:
print('Treasury Yields (if FRED is configured):')
print('(Requires FRED API key in config/api_keys.toml)')

try:
    # Try FRED provider for treasury rates
    from openbb import obb
    # Alternative: get treasury data from economy module
    result = obb.fixedincome.government.treasury_rates(provider='fred')
    rates = result.to_df() if result else pd.DataFrame()
    if not rates.empty:
        display(rates.head(10))
    else:
        print('  No treasury rate data.')
except Exception as e:
    print(f'  Not available: {e}')
    print('  → Get a free FRED API key: https://fred.stlouisfed.org/docs/api/api_key.html')

## 6. Economic Calendar (Upcoming Events)

In [ ]:
print('Upcoming Economic Events (if FMP is configured):')
print('(Requires FMP API key)')

try:
    calendar = obb.economy.calendar(provider='fmp')
    cal_df = calendar.to_df() if calendar else pd.DataFrame()
    if not cal_df.empty:
        display(cal_df.head(15))
    else:
        print('  No events data.')
except Exception as e:
    print(f'  Not available: {e}')

---
*Generated by AI Investment System*